# SEC 10-K Financial Analysis: Microsoft, Tesla, Apple (2023–2025)

**Author:** Junior Data Scientist, BCG GenAI Consulting Team
**Project:** AI-powered financial chatbot for Global Finance Corp. (GFC)

## Purpose

This notebook extracts insights from manually collected 10-K financial data
for Microsoft, Tesla, and Apple (fiscal years 2023–2025). The goal is to
identify significant financial trends and indicators that will later feed
into an AI-powered financial chatbot, so it can answer questions about
company performance and financial health.

**Metrics analyzed:** Total Revenue, Net Income, Total Assets, Total
Liabilities, and Cash Flow from Operating Activities.

**Data source:** [SEC EDGAR](https://www.sec.gov/edgar/search/) — figures
were extracted manually from each company's 10-K filings.

## Step 1: Setup

Import pandas and define the metrics we'll be working with.

In [ ]:
from pathlib import Path
import pandas as pd

METRICS = [
    "Revenue",
    "Net Income",
    "Total Assets",
    "Total Liabilities",
    "Operating Cash Flow",
]

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Step 2: Load the data

Loads the manually extracted, normalized filing data from CSV and validates
that all required columns are present. Replace `data/financial_data.csv`
with your own extracted figures — the expected columns are `Company`,
`Year`, and the five metrics above (all figures in the same units, e.g.
millions of USD).

In [ ]:
def load_data(path=None):
    """Load and validate the normalized filing data."""
    csv_path = Path(path) if path else Path("data") / "financial_data.csv"
    frame = pd.read_csv(csv_path)
    required = {"Company", "Year", *METRICS}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    return frame.sort_values(["Company", "Year"]).reset_index(drop=True)


df = load_data()
df

## Step 3: Calculate year-over-year growth and health indicators

For each company, we calculate:

- **Year-over-year (%) change** in Revenue, Net Income, and Operating Cash
  Flow — how fast each metric is growing or shrinking.
- **Net Margin (%)** — Net Income as a share of Revenue, i.e. profitability.
- **Operating Cash Flow Margin (%)** — how much of revenue converts into
  actual cash from operations.
- **Liabilities-to-Assets (%)** — a basic leverage/solvency indicator.

In [ ]:
def calculate_metrics(frame):
    """Add year-over-year changes and financial health indicators."""
    result = frame.copy()
    grouped = result.groupby("Company", sort=False)
    for metric in ["Revenue", "Net Income", "Operating Cash Flow"]:
        result[f"{metric} YoY (%)"] = grouped[metric].pct_change().mul(100).round(2)
    result["Net Margin (%)"] = result["Net Income"].div(result["Revenue"]).mul(100).round(2)
    result["Operating Cash Flow Margin (%)"] = (
        result["Operating Cash Flow"].div(result["Revenue"]).mul(100).round(2)
    )
    result["Liabilities to Assets (%)"] = (
        result["Total Liabilities"].div(result["Total Assets"]).mul(100).round(2)
    )
    return result


analyzed = calculate_metrics(df)
analyzed

## Step 4: Summarize overall change by company

Compares each company's first year (2023) to its last year (2025) across all
core metrics, to give a quick view of overall trajectory.

In [ ]:
def company_summary(frame):
    """Summarize first-to-last year changes by company."""
    ordered = frame.sort_values(["Company", "Year"])
    first = ordered.groupby("Company").first()
    last = ordered.groupby("Company").last()
    summary = pd.DataFrame(index=first.index)
    for metric in METRICS:
        summary[f"{metric} Change (%)"] = (
            last[metric].div(first[metric]).sub(1).mul(100).round(2)
        )
    return summary.reset_index()


summary = company_summary(analyzed)
summary

## Step 5: Visualize the trends

A quick look at revenue and net margin trends across companies makes it
easier to spot which companies are growing fastest and which are most
profitable — the kind of comparison the chatbot should be able to surface
in conversation.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for company, group in analyzed.groupby("Company"):
    axes[0].plot(group["Year"], group["Revenue"], marker="o", label=company)
    axes[1].plot(group["Year"], group["Net Margin (%)"], marker="o", label=company)

axes[0].set_title("Total Revenue by Year")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Revenue")
axes[0].legend()

axes[1].set_title("Net Margin (%) by Year")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Net Margin (%)")
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 6: Save results

Export the enriched dataset for downstream use (e.g. feeding the AI
chatbot).

In [ ]:
output_path = Path("data") / "financial_analysis_results.csv"
analyzed.to_csv(output_path, index=False)
print(f"Saved analysis results to {output_path}")

## Findings & Summary

*Replace this section with your own narrative once you've plugged in the
real extracted 10-K figures. Suggested points to cover:*

- **Revenue growth**: Which company grew fastest year-over-year? Any
  slowdowns or declines worth flagging?
- **Profitability (Net Margin)**: How do the three companies compare in
  converting revenue into profit? Is any company's margin trending down?
- **Cash generation**: Does Operating Cash Flow growth track with Net
  Income growth, or is there a gap worth investigating (e.g. working
  capital issues)?
- **Balance sheet health**: How does each company's Liabilities-to-Assets
  ratio compare? Is leverage increasing or decreasing over the period?
- **Implications for the chatbot**: What kinds of questions should the
  chatbot be able to answer based on this data (e.g. "How did Tesla's
  margin change from 2023 to 2025?" or "Which company has the strongest
  balance sheet?")?

## Methodology notes

- Figures were extracted manually from each company's 10-K filings on SEC
  EDGAR and normalized to consistent units (millions of USD) before
  loading into this notebook.
- Year-over-year percentages use `pandas.Series.pct_change()`, grouped by
  company so growth is never calculated across different companies.
- This analysis is a first step; before feeding data into the chatbot it
  should also be validated against the original filings and checked for
  outliers or restatements.